# 05 — Graph control, mapping, and serialization

**Objectives.** Compare SX/MX graphs, eliminate repeated Python loops, inspect functions, and persist graphs. Graph design often matters more than individual arithmetic operations.


In [ ]:
import casadi as ca
x = ca.MX.sym('x', 2); u = ca.MX.sym('u')
step = ca.Function('step', [x, u], [ca.vertcat(x[0] + 0.1*x[1], x[1] + 0.1*u)])
rollout = step.mapaccum('rollout', 8)
trajectory = rollout([0, 0], ca.DM.ones(1, 8))
assert trajectory.shape == (2, 8)
print(step.info(), 'nodes:', step.n_nodes())
serialized = step.serialize()
restored = ca.Function.deserialize(serialized)
assert float(ca.norm_inf(restored([0,0], 1) - step([0,0], 1))) == 0.0


`map` evaluates independent columns; `mapaccum` represents a recurrence. Both keep Python out of repeated evaluation and expose structure to CasADi. Function options such as `record_time`, `regularity_check`, and `post_expand` should be chosen deliberately and inspected with `print_options()` when needed.

**Common mistake:** a huge expanded graph can trade compilation and memory for a small evaluation speedup. Benchmark the actual workload. **Recap:** place reusable boundaries around models, stages, and residuals; map repeated structure; serialize only between compatible CasADi versions.
